# ROCLING 2026 DSA Baseline (Colab GPU)
中文 encoder + 雙回歸頭，預測新住民文本的 valence / arousal (1-9)。

**使用流程：**
1. 在本機先跑 `python prepare_data.py` 產生 `data/` 資料夾。
2. 把整個 `baseline/` 資料夾壓成 `baseline.zip`。
3. 上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**。
4. 依序執行下面每個 cell。訓練完會產生 `outputs/submission.csv` 可下載上傳。

In [ ]:
# 1) 安裝套件 (jieba 給 L1 詞典融合；gensim/sklearn/networkx 給 L2/L3)
!pip -q install "transformers>=4.40" torch numpy scipy scikit-learn jieba gensim networkx
import torch; print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2) 上傳 baseline.zip 並解壓 (裡面要含 train.py, prepare_data.py, data/)
from google.colab import files
up = files.upload()              # 選 baseline.zip
import zipfile, os
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z: z.extractall('.')
# 找出含 train.py 的資料夾並切進去
for root,_,fs in os.walk('.'):
    if 'train.py' in fs and 'data' in os.listdir(root):
        os.chdir(root); break
print('工作目錄:', os.getcwd()); print(os.listdir('.')); print('data:', os.listdir('data'))

In [ ]:
# 3) DAPT 領域適應續訓：在新住民文本上做 MLM (T4 上約幾分鐘)
# 若官方已釋出 test，存成含 text 欄的 csv 上傳，加 --extra <檔名> 語料更多更好
!python dapt.py --epochs 30 --batch_size 16 --model hfl/chinese-macbert-base

In [ ]:
# 4) 訓練。L1 詞典融合預設開啟（用 external/emobank 的 CVAW/CVAP）
# (L1) 不做 DAPT 的版本：
!python train.py --epochs 4 --batch_size 32 --model hfl/chinese-macbert-base
# (可選) 從 DAPT 權重接著微調：!python train.py --epochs 4 --batch_size 32 --model outputs/dapt_macbert
# (消融) 關掉 L1 詞典：加 --no_lexicon

In [ ]:
# 5) 檢視並下載 submission.csv
import pandas as pd
df = pd.read_csv('outputs/submission.csv')
print(df.describe()); df.head(10)

In [ ]:
from google.colab import files
files.download('outputs/submission.csv')